# Graph Caching

The slowest part of `OsmnxRouter` is downloading the street graph from
OpenStreetMap's Overpass API. For a 30-minute walking isochrone in a dense
city this can take 15–60 seconds. `GraphCache` stores the downloaded graph
to disk as a GraphML file keyed by `(lat, lon, radius, network_type)`.

A cache hit is a local file read — typically under 1 second.

**What's covered**
1. Creating a `GraphCache` and wiring it to `OsmnxRouter`
2. `IDD_GRAPH_CACHE_DIR` — activate caching via environment variable
3. Timing: uncached vs cached download
4. `has()`, `invalidate()`, `clear()` — cache management
5. Cache key scheme — why nearby origins share a graph
6. Best practices for batch and cluster use


## Setup

In [ ]:
import os
import tempfile
import time
from pathlib import Path

from idd_travel_time.constants import TransportMode
from idd_travel_time.io import GraphCache
from idd_travel_time.isochrone import compute
from idd_travel_time.models import TravelTimeRequest
from idd_travel_time.routers import OsmnxRouter

CACHE_DIR = Path(tempfile.mkdtemp()) / "graph_cache"
print(f"Cache directory: {CACHE_DIR}")

## 1 · Create a GraphCache and wire it to OsmnxRouter

In [ ]:
cache = GraphCache(CACHE_DIR)
print(cache)  # shows how many graphs are cached

router = OsmnxRouter(graph_cache=cache)

## 2 · First run — downloads from Overpass

In [ ]:
request = TravelTimeRequest(
    origin_lat=-1.2864,
    origin_lon=36.8172,
    mode=TransportMode.WALKING,
    duration_minutes=20,
)

t0 = time.monotonic()
result1 = compute(request, router=router)
elapsed_download = time.monotonic() - t0

print(f"First run (download): {elapsed_download:.1f}s")
print(cache)

## 3 · Second run — served from cache

In [ ]:
t0 = time.monotonic()
result2 = compute(request, router=router)
elapsed_cache = time.monotonic() - t0

print(f"Second run (cache):   {elapsed_cache:.1f}s")
print(f"Speedup:              {elapsed_download / elapsed_cache:.0f}×")

## 4 · `IDD_GRAPH_CACHE_DIR` — zero-code activation

Set the environment variable before starting your session and `OsmnxRouter`
picks it up automatically — no need to pass `graph_cache=` explicitly.


In [ ]:
os.environ["IDD_GRAPH_CACHE_DIR"] = str(CACHE_DIR)

# New router — no graph_cache= argument needed
auto_router = OsmnxRouter()
print(f"Cache auto-detected: {auto_router._cache}")
print(f"Cache dir: {auto_router._cache.cache_dir}")

In [ ]:
# Works exactly the same way
result3 = compute(request, router=auto_router)
print(f"Router: {result3.router_name}")

## 5 · Cache management

In [ ]:
# Check if a graph is cached
lat, lon = -1.2864, 36.8172

# Compute the radius that OsmnxRouter would use for this request
from idd_travel_time.routers._helpers import effective_speed_kph
from idd_travel_time.routers.osmnx_router import _travel_distance_meters

speed = effective_speed_kph(request)
dist = _travel_distance_meters(speed, request.duration_minutes)
radius_m = dist * 1.2  # 20% buffer

print(f"Effective radius: {radius_m:.0f} m")
print(f"Graph cached: {cache.has(lat, lon, radius_m)}")

In [ ]:
# List cached files
print("Cached graphs:")
for f in sorted(CACHE_DIR.glob("*.graphml")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}  ({size_kb:.0f} KB)")

In [ ]:
# Invalidate a specific graph (e.g. OSM data has changed)
deleted = cache.invalidate(lat, lon, radius_m)
print(f"Invalidated: {deleted}")
print(f"Now cached: {cache.has(lat, lon, radius_m)}")

In [ ]:
# Clear everything
n = cache.clear()
print(f"Cleared {n} graphs")
print(cache)

## 6 · Cache key scheme

Coordinates are rounded to 4 decimal places (~11 m). Two requests within
~11 m of each other share the same cached graph — useful when you have
slightly jittered origins for the same neighbourhood.


In [ ]:
from idd_travel_time.io.graph_cache import cache_key

# Two origins 11 m apart — same cache key
k1 = cache_key(-1.28640, 36.81720, 2000)
k2 = cache_key(-1.28641, 36.81721, 2000)
print(f"Within 11 m: same key? {k1 == k2}")
print(f"Key: {k1}")

# Two origins 110 m apart — different keys
k3 = cache_key(-1.28640, 36.81720, 2000)
k4 = cache_key(-1.28750, 36.81830, 2000)
print(f"\nBeyond 11 m: same key? {k3 == k4}")

## 7 · Batch and cluster best practices

```python
from idd_travel_time.io import GraphCache
from idd_travel_time.routers import OsmnxRouter
from idd_travel_time.isochrone import compute_parallel

# Shared cache on a network filesystem accessible to all workers
cache = GraphCache("/shared/nfs/graph_cache")

# Each worker process calls its own OsmnxRouter(graph_cache=cache)
# Per-key threading locks prevent duplicate downloads when two workers
# start at the same time for the same area.

results = compute_parallel(
    requests,
    router_factory=lambda: OsmnxRouter(graph_cache=cache),
    max_workers=8,
    output_dir="/shared/nfs/results",
    skip_existing=True,
)
```

**Key points:**
- One `GraphCache` instance per process is sufficient — it is not shared across processes, but each process reads the same files from disk.
- The per-key `threading.Lock` prevents duplicate downloads within a single process.
- For multi-process jobs, a second download may happen on first run if two workers race on the same key — the second writer simply overwrites the first with an identical file.
- Re-runs skip the download entirely because the file already exists.
